# Linear conjugate-gradient solver examples

## Preliminary steps

Importing necessary libraries

In [2]:
import sys,os
sys.path.append("/net/server/homes/sep/ettore/research/packages/pySolver/GenericSolver/python")
import pyVector as Vec
import pyOperator as Op
import pyLCGsolver as LCG
import pySymLCGsolver as SymLCGsolver
import pyProblem as Prblm
import pyStopperBase as Stopper
import sep_util as sep
import numpy as np
#Testing the NLCG solver to optimize a regularized linear problem treated as if it was non linear
import pyNLCGsolver as NLCG

Defining a matrix vector multiplication operator

In [3]:
class MatMult_incore(Op.Operator):
	"""Operator class to perform matrix-vector multiplication"""

	def __init__(self,A,domain,range):
		"""Constructor for the class: A = matrix to use; domain = domain vector; range = range vector"""
		if(not isinstance(domain,Vec.vector)): raise TypeError("ERROR! Domain vector not a vector object")
		if(not isinstance(range,Vec.vector)): raise TypeError("ERROR! Range vector not a vector object")
		#Setting domain and range of operator and matrix to use during application of the operator
		self.setDomainRange(domain,range)
		self.A = np.matrix(A)
		return

	def forward(self,add,model,data):
		"""Method to compute d = A m"""
		self.checkDomainRange(model,data)
		if(not isinstance(model,Vec.vectorIC)): raise TypeError("ERROR! Model vector not a vectorIC object")
		if(not isinstance(data,Vec.vectorIC)): raise TypeError("ERROR! Data vector not a vectorIC object")
		if(not add): data.zero()
		data.arr+=np.matmul(A,model.arr)
		return

	def adjoint(self,add,model,data):
		"""Method to compute m = A d"""
		self.checkDomainRange(model,data)
		if(not isinstance(model,Vec.vectorIC)): raise TypeError("ERROR! Model vector not a vectorIC object")
		if(not isinstance(data,Vec.vectorIC)): raise TypeError("ERROR! Data vector not a vectorIC object")
		if(not add): model.zero()
		model.arr+=np.matmul(A.H,data.arr)
		return

class MatMult_outcore(Op.Operator):
	"""Operator class to perform matrix-vector multiplication"""

	def __init__(self,A,domain,range):
		"""Constructor for the class: A = matrix to use; domain = domain vector; range = range vector"""
		if(not isinstance(domain,Vec.vector)): raise TypeError("ERROR! Domain vector not a vector object")
		if(not isinstance(range,Vec.vector)): raise TypeError("ERROR! Range vector not a vector object")
		#Setting domain and range of operator and matrix to use during application of the operator
		self.setDomainRange(domain,range)
		self.A = np.matrix(A)
		return

	def forward(self,add,model,data):
		"""Method to compute d = A m"""
		self.checkDomainRange(model,data)
		if(not isinstance(model,Vec.vectorOC)): raise TypeError("ERROR! Model vector not a vectorOC object")
		if(not isinstance(data,Vec.vectorOC)): raise TypeError("ERROR! Data vector not a vectorOC object")
		if(not add): data.zero()
		#Reading model and data vector files
		[model_arr,_]=sep.read_file(model.vecfile)
		[data_arr,data_axis]=sep.read_file(data.vecfile)
		data_arr+=np.matmul(A,model_arr)
		#writing data vector file
		sep.write_file(data.vecfile,data_arr,data_axis)
		return

	def adjoint(self,add,model,data):
		"""Method to compute m = A d"""
		self.checkDomainRange(model,data)
		if(not isinstance(model,Vec.vectorOC)): raise TypeError("ERROR! Model vector not a vectorOC object")
		if(not isinstance(data,Vec.vectorOC)): raise TypeError("ERROR! Data vector not a vectorOC object")
		if(not add): model.zero()
		#Reading model and data vector files
		[model_arr,model_axis]=sep.read_file(model.vecfile)
		[data_arr,_]=sep.read_file(data.vecfile)
		model_arr+=np.matmul(A.H,data_arr)
		#writing data vector file
		sep.write_file(model.vecfile,model_arr,model_axis)
		return

## Instantiation of vectors and operator

In [4]:
#In-core run
#Creating model vector
model_vec = Vec.vectorIC(np.zeros((100,1)))
model_vec.zero()
#Creating data vector
data_vec  = Vec.vectorIC(np.zeros((200,1)))
data_vec.rand()
#Matrix to be inverted
A = np.matrix(np.random.rand(200,100))
#Create operator
MatMult = MatMult_incore(A,model_vec,data_vec)

In [5]:
#Create L2-norm linear problem
L2Prob = Prblm.ProblemL2Linear(model_vec,data_vec,MatMult)
#Create stopper
niter = 2000
Stop  = Stopper.BasicStopper(niter=niter)
#Create solver
LCGsolver = LCG.LCGsolver(Stop)
#Running the solver
LCGsolver.run(L2Prob)

LINEAR CONJUGATE GRADIENT SOLVER
Restart folder: /net/storm/scr1/ettore/restart_2019-01-28T08-19-18.478444/

iter = 0 obj = 35.08394449791572 residual norm = 8.376627543100591 gradient norm= 67.88564267789482 feval = 2
iter = 1 obj = 34.57382941437998 residual norm = 8.315507129980706 gradient norm= 67.88564267789482 feval = 3
iter = 2 obj = 24.3410104171789 residual norm = 6.97725023446614 gradient norm= 23.324127452965303 feval = 5
iter = 3 obj = 21.857593739281263 residual norm = 6.611746174692622 gradient norm= 10.318949643410088 feval = 7
iter = 4 obj = 20.847131427660063 residual norm = 6.4571094814413765 gradient norm= 6.149808303877628 feval = 9
iter = 5 obj = 20.454342142677827 residual norm = 6.395989703349721 gradient norm= 3.645172683725204 feval = 11
iter = 6 obj = 20.254023370293666 residual norm = 6.364593210927729 gradient norm= 2.5811144772317838 feval = 13
iter = 7 obj = 20.161739460434095 residual norm = 6.350077079915502 gradient norm= 1.7963139972205382 feval = 15
